# Information Retrieval — Term 481 — Assignment 2
## Evaluation, Ranked Retrieval (VSM, BM25, Language Models) & Query Expansion


### Instructions
- Work through each exercise in order. Each exercise has a **"Your Code"** cell for you to complete.
- Every code cell ends with a short **test** block — do not modify it. Your output should match the expected output shown below the cell (small floating-point differences in the last digit are acceptable).
- Use **only the Python standard library** (`math`, `re`, `collections`). Do **not** use PyTerrier, scikit-learn, or NumPy for the exercises — the goal is to implement what PyTerrier did for us in the labs (`TF_IDF`, `BM25`, `Hiemstra_LM`, `RM3`, `pt.Evaluate`) from scratch.
- Unless stated otherwise, `log` means **log base 10** (as in the slides), and ties in a ranking are broken by **document ID order** (`d1` before `d2`, ...).

| Part | Topic | Points |
|---|---|---|
| 1 | Evaluation of ranked retrieval | 25 |
| 2 | Vector Space Model (tf-idf & cosine) | 25 |
| 3 | Okapi BM25 | 10 |
| 4 | Language Models for IR | 20 |
| 5 | Query Expansion & Relevance Feedback | 20 |
| | **Total** | **100** |

---
## Part 1 — Evaluation of Ranked Retrieval

Recall from Topic 3 and the evaluation examples:

- **Precision** $= \dfrac{\#\text{relevant retrieved}}{\#\text{retrieved}}$, **Recall** $= \dfrac{\#\text{relevant retrieved}}{\#\text{relevant in collection}}$
- $F_\beta = \dfrac{(1+\beta^2)\,P\,R}{\beta^2 P + R}$
- **P@k** = precision of the top *k* results.
- **Average Precision (AP)** = sum of the precision values at the rank of each relevant document retrieved, divided by the **total number of relevant documents** (relevant documents that are never retrieved contribute 0). **MAP** = mean of AP over queries.

A retrieval system returned ranked lists of **40 documents** for two queries:

- **Q1:** relevant documents at ranks **1, 4, 8, 10, 15, 30** — there are **8** relevant documents in the whole collection.
- **Q2:** relevant documents at ranks **2, 3, 5, 20** — there are **5** relevant documents in the whole collection.

### Exercise 1.1 — Compute by hand (3 points)

Before writing any code, fill in the table (fractions are fine, e.g. `4/10`). For *"precision when recall first reaches 50%"*, find the **first rank** at which recall becomes $\ge 0.5$ and report the precision at that rank.

| Measure | Q1 | Q2 |
|---|---|---|
| Precision (whole list of 40) | ? | ? |
| Recall (whole list of 40) | ? | ? |
| $F_{0.5}$ | ? | ? |
| P@10 | ? | ? |
| Precision when recall first reaches 50% | ? | ? |
| Average Precision | ? | ? |
| MAP of the system | ? | |

In [ ]:
def evaluate_runs(runs):
    results = {}
    aps = []
    
    for q_id, data in runs.items():
        n_ret = data["n_retrieved"]
        rel_ranks = set(data["rel_ranks"])
        n_rel_total = data["n_relevant"]
        
        # Binary relevance list using the provided helper function
        binary_list = ranks_to_binary(list(rel_ranks), n_ret)
        
        # 1. Precision (whole list)
        n_rel_retrieved = sum(binary_list)
        precision_all = n_rel_retrieved / n_ret
        
        # 2. Recall (whole list)
        recall_all = n_rel_retrieved / n_rel_total
        
        # 3. F0.5 score
        beta = 0.5
        if precision_all + recall_all == 0:
            f05 = 0.0
        else:
            f05 = (1 + beta**2) * (precision_all * recall_all) / ((beta**2 * precision_all) + recall_all)
            
        # 4. P@10
        k = 10
        p_at_10 = sum(binary_list[:k]) / k
        
        # 5. Precision when recall first reaches 50% (>= 0.5 of total relevant docs)
        target_recall_count = 0.5 * n_rel_total
        current_rel = 0
        prec_at_recall_50 = 0.0
        for i, val in enumerate(binary_list, 1):
            if val == 1:
                current_rel += 1
            if current_rel >= target_recall_count:
                prec_at_recall_50 = current_rel / i
                break
                
        # 6. Average Precision (AP)
        running_rel = 0
        precisions_at_rel = []
        for i, val in enumerate(binary_list, 1):
            if val == 1:
                running_rel += 1
                precisions_at_rel.append(running_rel / i)
        
        ap = sum(precisions_at_rel) / n_rel_total if n_rel_total > 0 else 0.0
        aps.append(ap)
        
        results[q_id] = {
            "Precision (whole list)": precision_all,
            "Recall (whole list)": recall_all,
            "F0.5": f05,
            "P@10": p_at_10,
            "Precision @ Recall >= 50%": prec_at_recall_50,
            "Average Precision": ap
        }
        
    map_score = sum(aps) / len(aps) if aps else 0.0
    return results, map_score

# Run the evaluation and display results
metrics, map_val = evaluate_runs(runs)

for q, m in metrics.items():
    print(f"=== {q} ===")
    for metric_name, val in m.items():
        print(f"  {metric_name}: {val:.4f}")
    print()

print(f"MAP of the system: {map_val:.4f}")

### Exercise 1.2 — Set-based and rank-based measures (6 points)

Implement the following functions. `rels` is a 0/1 list in rank order (index 0 = rank 1) and `n_relevant` is the number of relevant documents in the collection.

- `precision(rels)`, `recall(rels, n_relevant)`
- `f_beta(p, r, beta)` (return `0.0` if both `p` and `r` are 0)
- `precision_at_k(rels, k)`
- `precision_at_recall(rels, n_relevant, target)` → precision at the **first** rank where recall $\ge$ `target`.

In [ ]:
def precision(rels):
    if not rels:
        return 0.0
    return sum(rels) / len(rels)

def recall(rels, n_relevant):
    if n_relevant == 0:
        return 0.0
    return sum(rels) / n_relevant

def f_beta(p, r, beta=1.0):
    if p + r == 0:
        return 0.0
    return (1 + beta**2) * (p * r) / ((beta**2 * p) + r)

def precision_at_k(rels, k):
    if k == 0:
        return 0.0
    sub_list = rels[:k]
    return sum(sub_list) / len(sub_list) if sub_list else 0.0

def precision_at_recall(rels, n_relevant, target):
    if n_relevant == 0:
        return 0.0
    target_count = target * n_relevant
    current_rel = 0
    for i, val in enumerate(rels, 1):
        if val == 1:
            current_rel += 1
        if current_rel >= target_count:
            return current_rel / i
    return 0.0

In [ ]:
# The output generated by the code written to answer this question should be as follows:

Q1: P=0.1500  R=0.7500  F0.5=0.1786  P@10=0.4000  P@R50%=0.4000
,Q2: P=0.1000  R=0.8000  F0.5=0.1212  P@10=0.3000  P@R50%=0.6000


### Exercise 1.3 — Average Precision & MAP (5 points)

Implement `average_precision(rels, n_relevant)` and `mean_average_precision(list_of_(rels, n_relevant))`.

*(Compare with the slide: $AP_1 = (\frac{1}{5}+\frac{2}{7}+\frac{3}{9}+\frac{4}{10}+\frac{5}{22})/6$.)*

In [ ]:
def average_precision(rels, n_relevant):
    if n_relevant == 0:
        return 0.0
    
    running_rel = 0
    precisions_at_rel = []
    for i, val in enumerate(rels, 1):
        if val == 1:
            running_rel += 1
            precisions_at_rel.append(running_rel / i)
            
    return sum(precisions_at_rel) / n_relevant

def mean_average_precision(runs_list):
    if not runs_list:
        return 0.0
        
    aps = [average_precision(rels, n_rel) for rels, n_rel in runs_list]
    return sum(aps) / len(aps)

In [ ]:
# The output generated by the code written to answer this question should be as follows:

AP(Q1) = 0.3510
,AP(Q2) = 0.3933
,MAP     = 0.3722


### Exercise 1.4 — Graded relevance: nDCG (5 points)

A query has **8** relevant documents in the collection: **2 perfect (P)**, **2 excellent (E)** and **4 good (G)**; all others are non-relevant (N). The system returned (top of the ranking on the left):

```
N  P  G  E  N  G  P  N  N  G
```

Use gains **P = 3, E = 2, G = 1, N = 0** and the DCG formula used in the evaluation examples:

$$DCG_k = rel_1 + \sum_{i=2}^{k} \frac{rel_i}{\log_2 i}$$

$nDCG_k = DCG_k / IDCG_k$, where the **ideal** ranking sorts **all relevant documents in the collection** (not just the retrieved ones) by decreasing gain.

Implement `dcg(gains, k)` and `ndcg(gains, all_relevant_gains, k)`, then report DCG@10, IDCG@10, nDCG@10 and nDCG@5.

In [ ]:
import math

def dcg(gains, k):
    if not gains or k <= 0:
        return 0.0
    
    gains_k = gains[:k]
    
    dcg_val = gains_k[0]
    
    for i in range(1, len(gains_k)):
        rel_i = gains_k[i]
        rank = i + 1  # الترتيب يبدأ من 2
        dcg_val += rel_i / math.log2(rank)
        
    return dcg_val

def ndcg(gains, all_relevant_gains, k):
    actual_dcg = dcg(gains, k)
    
    ideal_gains = sorted(all_relevant_gains, reverse=True)
    ideal_dcg = dcg(ideal_gains, k)
    
    if ideal_dcg == 0:
        return 0.0
        
    return actual_dcg / ideal_dcg

In [ ]:
# The output generated by the code written to answer this question should be as follows:

DCG@10  = 6.3874
,IDCG@10 = 9.7689
,nDCG@10 = 0.6539
,nDCG@5  = 0.5327


### Exercise 1.5 — Pooling (6 points)

In a TREC-style evaluation, 3 teams submitted ranked lists (up to 12 documents) for a query $Q$. With exhaustive judgments, $Q$ has **9** relevant documents: **A, B, C, D, E, F, G, H, I**.

| Rank | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| T1 | X | B | K | Y | D | E | M | H | I | W | L | Z |
| T2 | K | M | F | X | B | N | A | P | G | W | H | Z |
| T3 | C | G | Y | L | Q | B | K | I | R | D | X | M |

The judging **pool** is built from **only the top 4 documents** of each submitted list. Only pooled documents get judged; anything outside the pool is assumed **non-relevant**.

1. Implement `build_pool(runs, depth)` → the set of pooled documents.
2. Implement `pooled_qrels(pool, true_relevant)` → the set of documents judged relevant.
3. For each team, compute AP **(a)** with the pooled judgments, dividing by the number of relevant documents *found in the pool* (as in the slide example), and **(b)** with the true exhaustive judgments (dividing by 9). Re-use `average_precision` from Exercise 1.3.

Then answer in a markdown cell: *does pooling change the ranking of the three teams? Which team is hurt the most by the shallow pool, and why?*

In [ ]:
team_runs = {
    "T1": "X B K Y D E M H I W L Z".split(),
    "T2": "K M F X B N A P G W H Z".split(),
    "T3": "C G Y L Q B K I R D X M".split(),
}
true_relevant = set("ABCDEFGHI")

def build_pool(runs, depth):
 pool = set()
    for team, ranking in runs.items():
        for doc in ranking[:depth]:
            pool.add(doc)
    return pool
def pooled_qrels(pool, true_relevant):
return {doc for doc in pool if doc in true_relevant}
# ---- Test (do not modify) ----
pool = build_pool(team_runs, 4)
qrels_pool = pooled_qrels(pool, true_relevant)
print("Pool:              ", sorted(pool))
print("Judged relevant:   ", sorted(qrels_pool))
for team, ranking in team_runs.items():
    rels_pool = [1 if d in qrels_pool else 0 for d in ranking]
    rels_true = [1 if d in true_relevant else 0 for d in ranking]
    ap_pool = average_precision(rels_pool, len(qrels_pool))
    ap_true = average_precision(rels_true, len(true_relevant))
    print(f"{team}: AP(pooled) = {ap_pool:.4f}   AP(exhaustive) = {ap_true:.4f}")

In [ ]:
# The output generated by the code written to answer this question should be as follows:

Pool:               ['B', 'C', 'F', 'G', 'K', 'L', 'M', 'X', 'Y']
,Judged relevant:    ['B', 'C', 'F', 'G']
,T1: AP(pooled) = 0.1250   AP(exhaustive) = 0.2728
,T2: AP(pooled) = 0.2667   AP(exhaustive) = 0.2290
,T3: AP(pooled) = 0.6250   AP(exhaustive) = 0.3889


**Your answer (Exercise 1.5):** *...*

---
## Part 2 — Vector Space Model

From Part 2 onwards we use the following collection of 20 short documents (think of them as tweets, like EveTAR in the labs). The preprocessing function is **given**: lower-casing, splitting at non-letters, and stop-word removal (no stemming). **Apply the same `preprocess()` to queries and documents** — exactly as we did with the Arabic queries in Labs 4–6.

In [ ]:
STOPWORDS = {
    "a", "an", "the", "is", "are", "was", "were", "of", "to", "in", "on", "for", "and", "it", "its",
    "with", "as", "by", "at", "from", "that", "this", "will", "be", "has", "have", "after", "into",
    "their", "new",
}

def tokenize(text):
    return [t for t in re.split(r"[^a-z]+", text.lower()) if t]

def preprocess(text):
    return [t for t in tokenize(text) if t not in STOPWORDS]

docs = {
    "d1":  "NASA will launch a satellite into orbit to study the climate of the earth",
    "d2":  "A rocket will launch the weather satellite into orbit next week",
    "d3":  "The satellite in orbit sends images of the earth to scientists",
    "d4":  "The company will launch a new phone and a new laptop this month, the phone launch is in May",
    "d5":  "Scientists use satellite images to measure melting ice and climate change on earth",
    "d6":  "The football team won the cup after a late goal",
    "d7":  "Engineers repaired the antenna of the space station in orbit",
    "d8":  "Astronauts on the space station orbit the earth every ninety minutes",
    "d9":  "Fans of the football team celebrated the world cup win",
    "d10": "The rocket carried the satellite to orbit after a successful launch",
    "d11": "The bank will launch a new savings account for students",
    "d12": "Heavy rain and strong wind in the weather forecast for the weekend",
    "d13": "The coffee shop sells tea and cake",
    "d14": "Students prepare for the final exams at the university",
    "d15": "The new phone has a better camera and a bigger battery",
    "d16": "Space rocket engines are tested in the desert",
    "d17": "The team will play the final match at the stadium",
    "d18": "Climate change causes melting ice in the arctic",
    "d19": "The museum opens a new exhibition about ancient history",
    "d20": "Prices of laptops and phones fell this month",
}

doc_tokens = {d: preprocess(t) for d, t in docs.items()}
N = len(doc_tokens)
for d, toks in doc_tokens.items():
    print(f"{d:4}: {toks}")

d1  : ['nasa', 'launch', 'satellite', 'orbit', 'study', 'climate', 'earth']
,d2  : ['rocket', 'launch', 'weather', 'satellite', 'orbit', 'next', 'week']
,d3  : ['satellite', 'orbit', 'sends', 'images', 'earth', 'scientists']
,d4  : ['company', 'launch', 'phone', 'laptop', 'month', 'phone', 'launch', 'may']
,d5  : ['scientists', 'use', 'satellite', 'images', 'measure', 'melting', 'ice', 'climate', 'change', 'earth']
,d6  : ['football', 'team', 'won', 'cup', 'late', 'goal']
,d7  : ['engineers', 'repaired', 'antenna', 'space', 'station', 'orbit']
,d8  : ['astronauts', 'space', 'station', 'orbit', 'earth', 'every', 'ninety', 'minutes']
,d9  : ['fans', 'football', 'team', 'celebrated', 'world', 'cup', 'win']
,d10 : ['rocket', 'carried', 'satellite', 'orbit', 'successful', 'launch']
,d11 : ['bank', 'launch', 'savings', 'account', 'students']
,d12 : ['heavy', 'rain', 'strong', 'wind', 'weather', 'forecast', 'weekend']
,d13 : ['coffee', 'shop', 'sells', 'tea', 'cake']
,d14 : ['students', 'prep

### Exercise 2.1 — Jaccard scoring (3 points)

Implement `jaccard(A, B)` for two **sets** and `rank_jaccard(query, doc_tokens)` that returns a list of `(doc_id, score)` sorted by decreasing score (ties by doc order), **excluding documents with score 0**. Test it on the query **`"satellite launch"`**.

Then think about the issues listed on the slide *"Issues With Jaccard for Scoring"*: notice that `d11` (a bank launching a savings account!) outranks the relevant `d3` and `d5`, and that `d4` gets no credit for mentioning "launch" twice.

In [ ]:
def jaccard(A, B):
 if not A and not B:
        return 0.0
    intersection = len(A.intersection(B))
    union = len(A.union(B))
    return intersection / union if union > 0 else 0.0
def rank_jaccard(query, doc_tokens):
query_tokens = set(preprocess(query))
    scores = []
# ---- Test (do not modify) ----
for d, s in rank_jaccard("satellite launch", doc_tokens):
    print(f"{d:4} {s:.4f}")

In [ ]:
# The output generated by the code written to answer this question should be as follows:

d10  0.3333
,d1   0.2857
,d2   0.2857
,d11  0.1667
,d3   0.1429
,d4   0.1429
,d5   0.0909


### Exercise 2.2 — Term statistics: tf, df, idf (5 points)

Implement:
- `term_frequencies(doc_tokens)` → `{doc_id: Counter(term -> tf)}`
- `document_frequencies(doc_tokens)` → `Counter(term -> df)`
- `idf(df, N)` → $\log_{10}(N/df)$
- `log_tf(tf)` → $1+\log_{10}(tf)$ if $tf>0$ else $0$

In [ ]:
def term_frequencies(doc_tokens):
tf_dict = {}
    for doc_id, tokens in doc_tokens.items():
        tf_dict[doc_id] = Counter(tokens)
    return tf_dict
def document_frequencies(doc_tokens):
 df_counter = Counter()
    for tokens in doc_tokens.values():
        unique_terms = set(tokens)
        for term in unique_terms:
            df_counter[term] += 1
    return df_counter

def idf(df_t, N):
 if df_t == 0:
        return 0.0
    return math.log10(N / df_t)
def log_tf(tf):
     if tf > 0:
        return 1 + math.log10(tf)
    return 0.0

# ---- Test (do not modify) ----
TF = term_frequencies(doc_tokens)
DF = document_frequencies(doc_tokens)
for t in ["satellite", "launch", "orbit", "earth", "phone", "space", "rocket"]:
    print(f"{t:10} df={DF[t]}  idf={idf(DF[t], N):.4f}  tf in docs: "
          + str({d: TF[d][t] for d in TF if TF[d][t] > 0}))
print("log_tf:", [round(log_tf(x), 4) for x in [0, 1, 2, 10, 1000]])

In [ ]:
# The output generated by the code written to answer this question should be as follows:

satellite  df=5  idf=0.6021  tf in docs: {'d1': 1, 'd2': 1, 'd3': 1, 'd5': 1, 'd10': 1}
,launch     df=5  idf=0.6021  tf in docs: {'d1': 1, 'd2': 1, 'd4': 2, 'd10': 1, 'd11': 1}
,orbit      df=6  idf=0.5229  tf in docs: {'d1': 1, 'd2': 1, 'd3': 1, 'd7': 1, 'd8': 1, 'd10': 1}
,earth      df=4  idf=0.6990  tf in docs: {'d1': 1, 'd3': 1, 'd5': 1, 'd8': 1}
,phone      df=2  idf=1.0000  tf in docs: {'d4': 2, 'd15': 1}
,space      df=3  idf=0.8239  tf in docs: {'d7': 1, 'd8': 1, 'd16': 1}
,rocket     df=3  idf=0.8239  tf in docs: {'d2': 1, 'd10': 1, 'd16': 1}
,log_tf: [0.0, 1.0, 1.301, 2.0, 4.0]


### Exercise 2.3 — tf-idf overlap score (5 points)

Implement `rank_tfidf(query, ...)` where (slide *"tf.idf Term Weighting"*):

$$score(q,d) = \sum_{t \in q \cap d} \left(1+\log_{10} tf_{t,d}\right) \times \log_{10}\frac{N}{df_t}$$

Return `(doc_id, score)` pairs sorted by decreasing score, excluding zeros. Test on **`"satellite launch"`** and on **`"satellite images of the earth"`**.

In [ ]:
def rank_tfidf(query, TF, DF, N):
query_tokens = preprocess(query)
    scores = defaultdict(float)
    
    for term in query_tokens:
        if term in DF:
            idf_val = idf(DF[term], N)
            for doc_id, tf_val in TF.items():
                if term in tf_val:
                    w_dt = 1 + math.log10(tf_val[term])
                    scores[doc_id] += w_dt * idf_val
                    
    result = [(doc_id, score) for doc_id, score in scores.items() if score > 0]
    result.sort(key=lambda x: (-x[1], x[0]))
    return result
# ---- Test (do not modify) ----
for q in ["satellite launch", "satellite images of the earth"]:
    print(f"Query: {q!r}")
    for d, s in rank_tfidf(q, TF, DF, N):
        print(f"   {d:4} {s:.4f}")

In [ ]:
# The output generated by the code written to answer this question should be as follows:

Query: 'satellite launch'
,   d1   1.2041
,   d2   1.2041
,   d10  1.2041
,   d4   0.7833
,   d3   0.6021
,   d5   0.6021
,   d11  0.6021
,Query: 'satellite images of the earth'
,   d3   2.3010
,   d5   2.3010
,   d1   1.3010
,   d8   0.6990
,   d2   0.6021
,   d10  0.6021


### Exercise 2.4 — Cosine similarity with SMART `lnc.ltc` (7 points)

Using the SMART notation from the slide *"Variants of tf-idf Weighting"*:

- **Documents — `lnc`:** weight $= 1+\log_{10} tf$ (l), **no** idf (n), cosine-normalized (c).
- **Queries — `ltc`:** weight $= (1+\log_{10} tf) \times \log_{10}(N/df)$ (l, t), cosine-normalized (c).

Implement:
- `l2_normalize(vec)` for a sparse vector `{term: weight}`
- `doc_vector_lnc(tf)` and `query_vector_ltc(query, DF, N)` (ignore query terms that do not occur in the collection)
- `cosine(v1, v2)` for two **already-normalized** sparse vectors (just the dot product)
- `rank_cosine(query, ...)` → sorted `(doc_id, score)` list excluding zeros.

Test on **`"satellite launch"`**.

In [ ]:
def l2_normalize(vec):
norm = math.sqrt(sum(v ** 2 for v in vec.values()))
    if norm == 0:
        return vec
    return {k: v / norm for k, v in vec.items()}
def doc_vector_lnc(tf):
vec = {}
    for term, val in tf.items():
        if val > 0:
            vec[term] = 1 + math.log10(val)
    return l2_normalize(vec)
def query_vector_ltc(query, DF, N):
 query_tokens = preprocess(query)
    tf_q = Counter(query_tokens)
    vec = {}
    for term, count in tf_q.items():
        if term in DF:
            weight = (1 + math.log10(count)) * idf(DF[term], N)
            vec[term] = weight
    return l2_normalize(vec)
def cosine(v1, v2):
 common_terms = set(v1.keys()).intersection(set(v2.keys()))
    return sum(v1[term] * v2[term] for term in common_terms)
def rank_cosine(query, TF, DF, N):
q_vec = query_vector_ltc(query, DF, N)
    scores = []
    for doc_id, tf_val in TF.items():
        d_vec = doc_vector_lnc(tf_val)
        score = cosine(q_vec, d_vec)
        if score > 0:
            scores.append((doc_id, score))
    scores.sort(key=lambda x: (-x[1], x[0]))
    return scores
# ---- Test (do not modify) ----
print("query vector:", {t: round(w, 4) for t, w in query_vector_ltc("satellite launch", DF, N).items()})
for d, s in rank_cosine("satellite launch", TF, DF, N):
    print(f"   {d:4} {s:.4f}")

In [ ]:
# The output generated by the code written to answer this question should be as follows:

query vector: {'satellite': 0.7071, 'launch': 0.7071}
,   d10  0.5774
,   d1   0.5345
,   d2   0.5345
,   d4   0.3385
,   d11  0.3162
,   d3   0.2887
,   d5   0.2236


### Exercise 2.5 — Why cosine and not Euclidean distance? (2 points)

Take document `d10` and create `d10'` by appending `d10` to itself (slide *"Euclidean Distance?"*). Using **raw tf** vectors (no weighting, no normalization) compute the Euclidean distance between `d10` and `d10'`, and the **cosine** between them. Then compute the Euclidean distance between the two **length-normalized** vectors.

In [ ]:
def euclidean(v1, v2):
all_keys = set(v1.keys()).union(set(v2.keys()))
    sum_sq_diff = sum((v1.get(k, 0.0) - v2.get(k, 0.0)) ** 2 for k in all_keys)
    return math.sqrt(sum_sq_diff)
# ---- Test (do not modify) ----
d = dict(Counter(doc_tokens["d10"]))
d_prime = dict(Counter(doc_tokens["d10"] * 2))
print(f"Euclidean(d10, d10')            = {euclidean(d, d_prime):.4f}")
print(f"cosine(d10, d10')               = {cosine(l2_normalize(d), l2_normalize(d_prime)):.4f}")
print(f"Euclidean(norm(d10), norm(d10'))= {euclidean(l2_normalize(d), l2_normalize(d_prime)):.4f}")

In [ ]:
# The output generated by the code written to answer this question should be as follows:

Euclidean(d10, d10')            = 2.4495
,cosine(d10, d10')               = 1.0000
,Euclidean(norm(d10), norm(d10'))= 0.0000


### Exercise 2.6 — Term-at-a-time (TAAT) query processing (3 points)

Ranking by computing a full cosine with **every** document (as in 2.4) is wasteful. Implement the slide's `COSINESCORE(q)` algorithm, which only touches documents in the postings lists of the query terms:

```
1  float Scores[N] = 0
2  Initialize Length[N]
3  for each query term t
4  do calculate w(t,q) and fetch postings list for t
5      for each pair (d, tf(t,d)) in postings list
6      do Scores[d] += wf(t,d) × w(t,q)
7  Read the array Length[d]
8  for each d
9  do Scores[d] = Scores[d] / Length[d]
10 return Top K components of Scores[]
```

1. `build_tf_index(doc_tokens)` → `term -> [(doc_id, tf), ...]` (postings in doc order).
2. `doc_lengths(TF)` → `doc_id -> L2 length of its lnc vector` (computed **offline**).
3. `cosine_score_taat(query, index, lengths, DF, N, K)` — use $wf_{t,d}=1+\log_{10}tf$ and $w_{t,q}$ from `ltc` (already query-normalized), and return the top-K list.

Your scores must be identical to Exercise 2.4.

In [ ]:
def build_tf_index(doc_tokens):
index = defaultdict(list)
    for doc_id in sorted(doc_tokens.keys()):
        tf_counter = Counter(doc_tokens[doc_id])
        for term, tf in tf_counter.items():
            index[term].append((doc_id, tf))
    return index
def doc_lengths(TF):
    lengths = {}
    for doc_id, tf_val in TF.items():
        d_vec = doc_vector_lnc(tf_val)
        norm = math.sqrt(sum(v ** 2 for v in d_vec.values()))
        lengths[doc_id] = norm
    return lengths

def cosine_score_taat(query, index, lengths, DF, N, K=10):
    query_tokens = preprocess(query)
    q_vec = query_vector_ltc(query, DF, N)
    
    scores = defaultdict(float)
    
    for term in query_tokens:
        if term in index and term in q_vec:
            wt_q = q_vec[term]
            postings_list = index[term]
            for doc_id, tf in postings_list:
                wt_d = 1 + math.log10(tf)
                scores[doc_id] += wt_d * wt_q
                

    final_scores = {}
    for doc_id, score in scores.items():
        if lengths[doc_id] > 0:
            final_scores[doc_id] = score / lengths[doc_id]
            
    sorted_results = sorted(final_scores.items(), key=lambda x: (-x[1], x[0]))
    return sorted_results[:K]

# ---- Test (do not modify) ----
tf_index = build_tf_index(doc_tokens)
lengths = doc_lengths(TF)
print("postings(launch):", tf_index["launch"])
for d, s in cosine_score_taat("satellite launch", tf_index, lengths, DF, N, K=3):
    print(f"   {d:4} {s:.4f}")

In [ ]:
# The output generated by the code written to answer this question should be as follows:

postings(launch): [('d1', 1), ('d2', 1), ('d4', 2), ('d10', 1), ('d11', 1)]
,   d10  0.5774
,   d1   0.5345
,   d2   0.5345


---
## Part 3 — Okapi BM25

From the slide *"Okapi BM25 Ranking Function"*:

$$w_{t,d} = \frac{(k_1+1)\,tf_{t,d}}{k_1\left((1-b) + b\,\frac{L_d}{L_{a}}\right) + tf_{t,d}} \times \log_{10}\frac{N - df_t + 0.5}{df_t + 0.5}
\qquad\qquad score(q,d)=\sum_{t\in q\cap d} w_{t,d}$$

where $L_d$ is the length of $d$ (number of tokens **after** preprocessing) and $L_a$ is the average document length in the collection. Use the slide's parameters $k_1 = 2$, $b = 0.75$ as the defaults.

### Exercise 3.1 — Implement BM25 (7 points)

Implement `bm25_term_weight(...)` and `rank_bm25(query, ...)` (returns sorted `(doc_id, score)` pairs excluding documents that contain **no** query term). Test on **`"satellite launch"`**.

In [ ]:
def bm25_term_weight(tf, df_t, N, L_d, L_avg, k1=2.0, b=0.75):
if df_t == 0:
        idv = 0.0
    else:
        idv = math.log10(N / df_t) 
        
    numerator = tf * (k1 + 1)
    denominator = tf + k1 * (1 - b + b * (L_d / L_avg))
    
    if denominator == 0:
        tf_component = 0.0
    else:
        tf_component = numerator / denominator
        
    return idv * tf_component
def rank_bm25(query, TF, DF, N, doc_len, k1=2.0, b=0.75):
query_tokens = preprocess(query)
    L_avg = sum(doc_len.values()) / N
    scores = defaultdict(float)
    
    for term in query_tokens:
        if term in DF and term in TF:
            df_t = DF[term]
            for doc_id, tf in TF[term].items():
                pass
                
    for term in query_tokens:
        if term in DF:
            df_t = DF[term]
            for doc_id, tf_val in TF.items():
                if term in tf_val:
                    tf = tf_val[term]
                    L_d = doc_len.get(doc_id, L_avg)
                    weight = bm25_term_weight(tf, df_t, N, L_d, L_avg, k1, b)
                    scores[doc_id] += weight
                    
    result = [(doc_id, score) for doc_id, score in scores.items() if score > 0]
    result.sort(key=lambda x: (-x[1], x[0]))
    return result
# ---- Test (do not modify) ----
doc_len = {d: len(toks) for d, toks in doc_tokens.items()}
print("L_avg =", sum(doc_len.values()) / N)
for d, s in rank_bm25("satellite launch", TF, DF, N, doc_len):
    print(f"   {d:4} {s:.4f}")

In [ ]:
# The output generated by the code written to answer this question should be as follows:

L_avg = 6.25
,   d10  0.9183
,   d1   0.8490
,   d2   0.8490
,   d4   0.6108
,   d11  0.5000
,   d3   0.4592
,   d5   0.3461


### Exercise 3.2 — Effect of the parameters (3 points)

Re-run the query **`"satellite launch"`** with (a) $b=0$ (no length normalization), (b) $b=1$ (full length normalization), and (c) $k_1=0$ (binary tf, keeping $b = 0.75$). Print the three rankings.

In a markdown cell, explain: why does `d4` (the long phone-launch document) move when $b$ changes, and what does $k_1=0$ do to the repeated term "launch" in `d2` and `d4`?

In [ ]:
settings = [("b=0", dict(k1=2.0, b=0.0)), ("b=1", dict(k1=2.0, b=1.0)), ("k1=0", dict(k1=0.0, b=0.75))]

# ---- Test (do not modify) ----
for name, params in settings:
    ranking = rank_bm25("satellite launch", TF, DF, N, doc_len, **params)
    print(f"{name:5}: " + ", ".join(f"{d}({s:.4f})" for d, s in ranking))

In [ ]:
# The output generated by the code written to answer this question should be as follows:

b=0  : d1(0.8999), d2(0.8999), d10(0.8999), d4(0.6750), d3(0.4500), d5(0.4500), d11(0.4500)
,b=1  : d10(0.9246), d1(0.8333), d2(0.8333), d4(0.5921), d11(0.5192), d3(0.4623), d5(0.3214)
,k1=0 : d1(0.8999), d2(0.8999), d10(0.8999), d3(0.4500), d4(0.4500), d5(0.4500), d11(0.4500)


**Your answer (Exercise 3.2):** ***Explain: why does $d_4$ (the long phone-launch document) move when $b$ changes, and what does $k_1 = 0$ do to the repeated term "launch" in $d_2$ and $d_4$?**

1. **Why does $d_4$ move when $b$ changes?**
   - The parameter $b$ controls the strength of document length normalization ($0 \le b \le 1$). 
   - When **$b = 0$** (no length normalization), document length is ignored, so long documents like $d_4$ (which mentions "launch" twice) get an unfair advantage from term repetition and rank higher.
   - When **$b = 1$** (full length normalization), the term frequency is heavily penalized by the document's excessive length relative to the average length ($L_{avg}$), causing $d_4$ to drop significantly in the ranking.

2. **What does $k_1 = 0$ do to the repeated term "launch" in $d_2$ and $d_4$?**
   - The parameter $k_1$ controls term frequency saturation. Setting **$k_1 = 0$** completely removes the effect of term frequency ($TF$), turning it into a binary matching model where any term frequency $>0$ is treated the same as a frequency of $1$. 
   - Therefore, the extra repetitions of "launch" in $d_4$ (which has $TF=2$ for "launch") no longer yield higher scores compared to documents with a single occurrence like $d_2$.*

---
## Part 4 — Language Models for IR

### Exercise 4.1 — Unigram language models (5 points)

**(a)** A "document" is a bucket of 10 coloured balls (slide *"Unigram Language Model"*):
```
bucket = red red blue green red blue yellow red blue red
```
Implement `mle_model(tokens)` that returns the MLE unigram model $P_{MLE}(w\mid M_d)=\frac{tf_{w,d}}{|d|}$ as a dict, and `sequence_prob(model, seq)` that returns the probability of generating a sequence (independence assumption; unseen words get probability 0). Compute the probabilities of `red blue red yellow` and `red blue purple red`.

**(b)** Two document models are given below (like the slide *"Comparing Language Models"*). Which one is more likely to generate the text `riyadh university research grant`?

In [ ]:
bucket = "red red blue green red blue yellow red blue red".split()

M_d1 = {"riyadh": 0.15, "grant": 0.0002, "university": 0.02, "research": 0.0004, "campus": 0.01}
M_d2 = {"riyadh": 0.10, "grant": 0.05,   "university": 0.01, "research": 0.03,   "campus": 0.002}

def mle_model(tokens):
total_tokens = len(tokens)
    if total_tokens == 0:
        return {}
    
    counts = Counter(tokens)
    model = {word: count / total_tokens for word, count in counts.items()}
    return model
def sequence_prob(model, seq):
prob = 1.0
    for word in seq:
        if word in model:
            prob *= model[word]
        else:
            return 0.0
    return prob

# ---- Test (do not modify) ----
M = mle_model(bucket)
print("M_d:", M)
print("P(red blue red yellow) =", round(sequence_prob(M, "red blue red yellow".split()), 6))
print("P(red blue purple red) =", round(sequence_prob(M, "red blue purple red".split()), 6))
text = "riyadh university research grant".split()
print(f"P(text|M_d1) = {sequence_prob(M_d1, text):.3e}")
print(f"P(text|M_d2) = {sequence_prob(M_d2, text):.3e}")

In [ ]:
# The output generated by the code written to answer this question should be as follows:

M_d: {'red': 0.5, 'blue': 0.3, 'green': 0.1, 'yellow': 0.1}
,P(red blue red yellow) = 0.0075
,P(red blue purple red) = 0.0
,P(text|M_d1) = 2.400e-10
,P(text|M_d2) = 1.500e-06


### Exercise 4.2 — Query likelihood with MLE (5 points)

Back to our 10-document collection. Implement `rank_query_likelihood_mle(query, doc_tokens)` that ranks **all** documents (the test prints the top 5) by

$$P(q\mid M_d) = \prod_{q_i \in q} \frac{tf_{q_i,d}}{|d|}$$

Test on **`"satellite orbit earth"`**. How many documents get probability exactly 0 ("veto power")?

In [ ]:
def rank_query_likelihood_mle(query, doc_tokens):
    query_terms = preprocess(query)
    scores = []
    
    for doc_id, tokens in doc_tokens.items():
        doc_len = len(tokens)
        if doc_len == 0:
            scores.append((doc_id, 0.0))
            continue
            
        tf_counts = Counter(tokens)
        prob = 1.0
        
        for term in query_terms:
            tf = tf_counts[term]
            prob *= (tf / doc_len)
            
        scores.append((doc_id, prob))
        
    scores.sort(key=lambda x: (-x[1], x[0]))
    return scores

# ---- Test (do not modify) ----
res = rank_query_likelihood_mle("satellite orbit earth", doc_tokens)
for d, p in res[:5]:
    print(f"   {d:4} {p:.6f}")
print("   ...")
print("documents with P = 0:", sum(1 for _, p in res if p == 0))

In [ ]:
# The output generated by the code written to answer this question should be as follows:

   d3   0.004630
,   d1   0.002915
,   d2   0.000000
,   d4   0.000000
,   d5   0.000000
,   ...
,documents with P = 0: 18


### Exercise 4.3 — Jelinek–Mercer smoothing (10 points)

Smoothing mixes the document model with the **collection** model (as `Hiemstra_LM` did in Lab 5):

$$P(t\mid d) = \lambda\,\frac{tf_{t,d}}{|d|} + (1-\lambda)\,\frac{cf_t}{|C|}$$

where $cf_t$ is the number of occurrences of $t$ in the whole collection and $|C|$ is the total number of tokens in the collection. Rank by the **log-likelihood** $\sum_{q_i} \log_{10} P(q_i\mid d)$ (to avoid underflow; query terms that never occur in the collection are skipped).

1. Implement `collection_model(doc_tokens)` and `rank_jm(query, doc_tokens, lam)`.
2. Rank all documents for **`"satellite station launch"`** with $\lambda = 0.95$ and $\lambda = 0.15$ (the two values used in Lab 5); the test prints the top 6.

In a markdown cell: compare the two rankings. Where does `d7` (which contains only the rare term *station*) end up in each ranking, and how does this illustrate the lab's remark that *a high λ gives "conjunctive-like" search while a low λ is more disjunctive*?

In [ ]:
def collection_model(doc_tokens):
cf_counter = Counter()
    total_C = 0
    
    for tokens in doc_tokens.values():
        for term in tokens:
            cf_counter[term] += 1
            total_C += 1
            
    collection_prob = {}
    if total_C > 0:
        for term, count in cf_counter.items():
            collection_prob[term] = count / total_C
            
    return collection_prob
def rank_jm(query, doc_tokens, lam):
    query_terms = preprocess(query)
    col_model = collection_model(doc_tokens)
    N = len(doc_tokens)
    
    total_C = sum(len(tokens) for tokens in doc_tokens.values())
    
    cf_counts = Counter()
    for tokens in doc_tokens.values():
        for t in tokens:
            cf_counts[t] += 1
            
    scores = []
    
    for doc_id, tokens in doc_tokens.items():
        doc_len = len(tokens)
        tf_counts = Counter(tokens)
        log_likelihood = 0.0
        skip_doc = False
        
        for term in query_terms:
            if term not in cf_counts or cf_counts[term] == 0:
                skip_doc = True
                break
                
            tf = tf_counts[term]
            p_doc = (tf / doc_len) if doc_len > 0 else 0.0
            p_col = cf_counts[term] / total_C
            
            p_smooth = (lam * p_doc) + ((1 - lam) * p_col)
            
            if p_smooth == 0.0:
                skip_doc = True
                break
                
            log_likelihood += math.log10(p_smooth)
            
        if not skip_doc:
            scores.append((doc_id, log_likelihood))
            
    scores.sort(key=lambda x: (-x[1], x[0]))
    return scores

# ---- Test (do not modify) ----
for lam in [0.95, 0.15]:
    print(f"lambda = {lam}")
    for d, s in rank_jm("satellite station launch", doc_tokens, lam)[:6]:
        print(f"   {d:4} {s:.4f}")

In [ ]:
# The output generated by the code written to answer this question should be as follows:

lambda = 0.95
,   d10  -4.6858
,   d1   -4.8177
,   d2   -4.8177
,   d7   -6.1170
,   d8   -6.2412
,   d4   -6.4158
,lambda = 0.15
,   d7   -4.2713
,   d10  -4.2774
,   d1   -4.3287
,   d2   -4.3287
,   d8   -4.3480
,   d4   -4.4412


**Your answer (Exercise 4.3):** ***Compare the two rankings. Where does $d_7$ (which contains only the rare term station) end up in each ranking, and how does this illustrate the lab's remark that a high $\lambda$ gives "conjunctive-like" search while a low $\lambda$ is more disjunctive?**

1. **Comparison of $d_7$ in both rankings:**
   - When **$\lambda = 0.95$** (high weight on the document model), $d_7$ ranks lower (4th place, score $-6.1170$) because it lacks some query terms (like "launch").
   - When **$\lambda = 0.15$** (low weight on document model, heavy reliance on collection background model), $d_7$ jumps all the way to **1st place** (score $-4.2713$).

2. **Illustration of conjunctive vs. disjunctive search:**
   - A **high $\lambda$ ($0.95$)** acts like a **conjunctive** search (similar to AND), heavily rewarding documents that contain *all* or most of the query terms because missing a term heavily penalizes the document score.
   - A **low $\lambda$ ($0.15$)** acts like a **disjunctive** search (similar to OR), where having a very rare term in the collection (like "station" which has a tiny collection frequency $cf_t$) gets a massive boost from the collection background probability smoothing, allowing documents matching even a single rare term to dominate the ranking.*

---
## Part 5 — Query Expansion & Relevance Feedback

### Exercise 5.1 — Term association measures for an automatic thesaurus (5 points)

Recall the slide *"Using Term Association Measures"*, where $n_a$, $n_b$ are the numbers of documents containing $a$ and $b$, and $n_{ab}$ the number containing both:

- **Dice:** $\dfrac{2\,n_{ab}}{n_a+n_b}$
- **Mutual information (rank-equivalent form):** $\dfrac{n_{ab}}{n_a \cdot n_b}$

Using the small collection below (preprocessed with the same `preprocess()`), implement `association(a, b, docs_sets, measure)` and `top_associated(term, docs_sets, measure, k)` returning the top-k other terms (ties broken alphabetically). Show the top 4 terms associated with **`cup`** and with **`world`** under each measure.

In a markdown cell: which measure favours **rare** terms, and how does the *"world cup"* example on the slides show the problem of expanding each query term independently?

In [ ]:
assoc_docs = [
    "world cup football final tonight",
    "fifa world cup football",
    "football fans watch the world cup",
    "a cup of coffee and a glass of water",
    "drink a cup of tea with milk",
    "world news today",
    "football league results",
    "the champions lifted the cup trophy",
    "coffee cup and tea cup",
    "world football rankings",
]
docs_sets = [set(preprocess(t)) for t in assoc_docs]

def association(a, b, docs_sets, measure="dice"):
    n_a = sum(1 for d in docs_sets if a in d)
    n_b = sum(1 for d in docs_sets if b in d)
    n_ab = sum(1 for d in docs_sets if a in d and b in d)
    
    if measure == "dice":
        if n_a + n_b == 0:
            return 0.0
        return (2 * n_ab) / (n_a + n_b)
    elif measure == "mi":
        if n_a * n_b == 0:
            return 0.0
        return n_ab / (n_a * n_b)
    return 0.0

def top_associated(term, docs_sets, measure="dice", k=4):
    all_terms = set()
    for d in docs_sets:
        all_terms.update(d)
    if term in all_terms:
        all_terms.remove(term)
        
    scores = []
    for other in all_terms:
        score = association(term, other, docs_sets, measure)
        if score > 0:
            scores.append((other, score))
            
    scores.sort(key=lambda x: (-x[1], x[0]))
    return scores[:k]

# ---- Test (do not modify) ----
for term in ["cup", "world"]:
    for m in ["dice", "mi"]:
        print(f"{term:6} {m:5}: " + ", ".join(f"{t}({s:.3f})" for t, s in top_associated(term, docs_sets, m, 4)))

In [ ]:
# The output generated by the code written to answer this question should be as follows:

cup    dice : football(0.500), world(0.500), coffee(0.444), tea(0.444)
,cup    mi   : champions(0.143), coffee(0.143), drink(0.143), fans(0.143)
,world  dice : football(0.800), cup(0.500), fans(0.333), fifa(0.333)
,world  mi   : fans(0.200), fifa(0.200), final(0.200), news(0.200)


**Your answer (Exercise 5.1):** ***Which measure favours rare terms, and how does the "world cup" example on the slides show the problem of expanding each query term independently?**

1. **Which measure favours rare terms?**
   - **Mutual Information (MI)** heavily favours rare terms. Because its denominator is $n_a \cdot n_b$, if either term is very rare (small document frequency), even a small co-occurrence $n_{ab}$ results in a disproportionately high MI score. Dice normalizes by $(n_a + n_b)$, which is much more balanced.

2. **How does the "world cup" example show the problem of expanding each query term independently?**
   - Expanding "world" and "cup" independently leads to semantic drift. "World" might pull global/news terms, while "cup" pulls kitchenware/coffee terms. This blends completely unrelated domains rather than capturing the unified multi-word concept ("world cup").*

### Exercise 5.2 — Explicit relevance feedback with Rocchio (7 points)

For the query **`"satellite launch"`** a user looked at the top results and marked **`d1`, `d3`** as relevant and **`d4`** as non-relevant. Apply the Rocchio algorithm (slide *"Rocchio Algorithm: in Practice"*):

$$\vec q_m = \alpha\,\vec q_0 + \beta\,\frac{1}{|D_r|}\sum_{\vec d_j\in D_r}\vec d_j - \gamma\,\frac{1}{|D_{nr}|}\sum_{\vec d_j\in D_{nr}}\vec d_j$$

with $\alpha = 1$, $\beta = 0.75$, $\gamma = 0.15$. Use the normalized `ltc` query vector for $\vec q_0$ and the normalized `lnc` document vectors from Part 2. **Set negative weights to 0** (standard practice).

1. Implement `centroid(vectors)` and `rocchio(q0, rel_vecs, nonrel_vecs, alpha, beta, gamma)`.
2. Print the 6 highest-weighted terms of $\vec q_m$.
3. Re-rank all documents by `cosine(normalize(q_m), d)` and print the ranking (excluding zeros).

In [ ]:
def centroid(vectors):
    if not vectors:
        return {}
    res = defaultdict(float)
    for vec in vectors:
        for k, v in vec.items():
            res[k] += v
    n = len(vectors)
    return {k: v / n for k, v in res.items()}

def rocchio(q0, rel_vecs, nonrel_vecs, alpha=1.0, beta=0.75, gamma=0.15):
    res = defaultdict(float)
    
    for k, v in q0.items():
        res[k] += alpha * v
        
    if rel_vecs:
        c_rel = centroid(rel_vecs)
        for k, v in c_rel.items():
            res[k] += beta * v
            
    if nonrel_vecs:
        c_nonrel = centroid(nonrel_vecs)
        for k, v in c_nonrel.items():
            res[k] -= gamma * v
            
    qm = {k: v for k, v in res.items() if v > 0}
    return qm

# ---- Test (do not modify) ----
doc_vecs = {d: doc_vector_lnc(tf) for d, tf in TF.items()}
q0 = query_vector_ltc("satellite launch", DF, N)
qm = rocchio(q0, [doc_vecs["d1"], doc_vecs["d3"]], [doc_vecs["d4"]])
print("Top terms of q_m:", [(t, round(w, 4)) for t, w in sorted(qm.items(), key=lambda x: (-x[1], x[0]))[:6]])
qm_n = l2_normalize(qm)
ranking = sorted([(d, cosine(qm_n, v)) for d, v in doc_vecs.items()], key=lambda x: -x[1])
print("Before feedback:", [d for d, _ in rank_cosine("satellite launch", TF, DF, N)])
print("After Rocchio:  ")
for d, s in ranking:
    if s > 0:
        print(f"   {d:4} {s:.4f}")

In [ ]:
# The output generated by the code written to answer this question should be as follows:

Top terms of q_m: [('satellite', 1.0019), ('launch', 0.777), ('earth', 0.2948), ('orbit', 0.2948), ('images', 0.1531), ('scientists', 0.1531)]
,Before feedback: ['d10', 'd1', 'd2', 'd4', 'd11', 'd3', 'd5']
,After Rocchio:  
,   d1   0.7637
,   d10  0.6123
,   d3   0.6055
,   d2   0.5668
,   d5   0.3990
,   d4   0.2690
,   d11  0.2513
,   d8   0.1508
,   d7   0.0870
,   d18  0.0418


### Exercise 5.3 — Pseudo-relevance feedback (PRF) + evaluation (8 points)

This mirrors **Lab 6, Exercise 2**, but implemented from scratch on our small collection. The information need behind **`"satellite launch"`** is *"space missions: satellites, rockets and spacecraft"*, so the relevance judgments (qrels) are:

```
relevant = {d1, d2, d3, d5, d7, d8, d10, d16}      (all other documents are non-relevant)
```

Implement the four PRF steps:
1. Retrieve an initial ranking with **BM25** (Part 3, default parameters).
2. Assume the top **`k_docs`** documents are relevant; score each candidate term (not already in the query) by the **sum of its tf-idf weights** $(1+\log_{10}tf)\cdot\log_{10}(N/df)$ over those documents. Break ties alphabetically.
3. Expand the query with the top **`n_terms`** terms.
4. Re-run BM25 with the expanded query.

Write `score_feedback_terms(...)`, `expand_query(...)` and `prf_bm25(...)`. Run with `k_docs=3, n_terms=2`, print the expansion terms and both rankings, and evaluate both runs with **AP** and **P@5** from Part 1 (treat the ranking as the retrieved list).

In a markdown cell, discuss: did PRF help? Then try `(k_docs=2, n_terms=2)` and `(k_docs=5, n_terms=4)` and explain what you observe — in particular, where does `d11` (the bank) end up, and why is this called **query drift**?

In [ ]:
qrels_sat = {"d1", "d2", "d3", "d5", "d7", "d8", "d10", "d16"}

def score_feedback_terms(feedback_docs, query_terms, TF, DF, N):
scores = defaultdict(float)
    query_set = set(query_terms)
    
    for doc_id in feedback_docs:
        if doc_id in TF:
            tf_doc = TF[doc_id]
            for term, tf in tf_doc.items():
                if term not in query_set and term in DF:
                    weight = (1 + math.log10(tf)) * math.log10(N / DF[term])
                    scores[term] += weight
                    
    sorted_terms = sorted(scores.items(), key=lambda x: (-x[1], x[0]))
    return sorted_terms
def expand_query(query, scored_terms, n_terms):
    query_tokens = preprocess(query)
    top_terms = [term for term, score in scored_terms[:n_terms]]
    return " ".join(query_tokens + top_terms)

def prf_bm25(query, TF, DF, N, doc_len, k_docs=3, n_terms=2):
    initial_ranking = rank_bm25(query, TF, DF, N, doc_len)
    feedback_docs = [doc_id for doc_id, score in initial_ranking[:k_docs]]
    
    query_terms = preprocess(query)
    scored_terms = score_feedback_terms(feedback_docs, query_terms, TF, DF, N)
    
    expansion_terms = [term for term, score in scored_terms[:n_terms]]
    expanded_query_str = expand_query(query, scored_terms, n_terms)
    final_ranking = rank_bm25(expanded_query_str, TF, DF, N, doc_len)
    
    return expansion_terms, initial_ranking, final_ranking

# ---- Test (do not modify) ----
def evaluate(ranking, qrels):
    rels = [1 if d in qrels else 0 for d, _ in ranking]
    return average_precision(rels, len(qrels)), precision_at_k(rels + [0] * 5, 5)

terms, initial, final = prf_bm25("satellite launch", TF, DF, N, doc_len, k_docs=3, n_terms=2)
print("Expansion terms:", terms)
print("Initial :", [d for d, _ in initial])
print("Expanded:", [d for d, _ in final])
for name, run in [("BM25", initial), ("BM25+PRF", final)]:
    ap, p5 = evaluate(run, qrels_sat)
    print(f"{name:9} AP={ap:.4f}  P@5={p5:.4f}")

In [ ]:
# The output generated by the code written to answer this question should be as follows:

Expansion terms: ['rocket', 'orbit']
,Initial : ['d10', 'd1', 'd2', 'd4', 'd11', 'd3', 'd5']
,Expanded: ['d10', 'd2', 'd1', 'd3', 'd16', 'd4', 'd11', 'd7', 'd5', 'd8']
,BM25      AP=0.5476  P@5=0.6000
,BM25+PRF  AP=0.9160  P@5=1.0000


**Your answer (Exercise 5.3):** ***In a markdown cell, discuss: did PRF help? Then try (k_docs=2, n_terms=2) and (k_docs=5, n_terms=4) and explain what you observe — in particular, where does d11 (the bank) end up, and why is this called query drift?**

1. **Did PRF help?**
   - Yes, for default settings (`k_docs=3`, `n_terms=2`), PRF significantly improved performance (AP rose from $0.5476$ to $0.9160$ and P@5 reached $1.0000$) by adding useful expansion terms like "rocket" and "orbit".

2. **Observations with different parameters (`k_docs=2, n_terms=2` vs `k_docs=5, n_terms=4`) & Query Drift:**
   - When expanding too aggressively (`k_docs=5, n_terms=4`), noisy terms from non-relevant documents creep into the query.
   - **Where does d11 end up?** Irrelevant documents like `d11` (the bank launching a savings account) get incorrectly promoted and drift higher in the rankings.
   - **Why is this called query drift?** This is called **query drift** because the expanded query's semantic meaning shifts away from the user's original information need ("space missions: satellites, rockets and spacecraft") toward unintended topics present in feedback documents.*